# Silver Corrections — Auto-fix + Manual Override

**Scop:** Procesarea înregistrărilor din `banking.silver_quarantine.qrt_*` prin:
1. **Auto-fix** — corecții deterministe automate (carduri expirate, sume negative, etc.)
2. **Manual override** — corecții explicite scrise de echipa de date în tabele dedicate

**Pattern de migrare:** marcare în quarantine cu flag `fix_status` + copiere în Silver dim/fact.

**Stari posibile pentru un rând quarantine:**
- `pending` — încă nu a fost procesat (default după Silver)
- `auto_fixed` — reparat automat și migrat în Silver
- `manual_fixed` — reparat prin override manual și migrat în Silver
- `unfixable` — nu a putut fi reparat (rămâne în quarantine pentru analiză)

**Audit trail:** fiecare rulare loghează în `silver_corrections.fix_log` câte rânduri au fost încercate, fixate și ratate.

## 1. Configurare și utilitare

In [0]:
from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, when, lit, current_timestamp, to_date, to_timestamp,
    array, size, concat_ws, abs as spark_abs,
    coalesce, expr, broadcast,
    filter as array_filter,
)
from pyspark.sql.types import (
    IntegerType, DoubleType, StringType, TimestampType, DateType, StructType, StructField
)
from delta.tables import DeltaTable

CATALOG       = "banking"
SILVER        = "silver"
QUARANTINE    = "silver_quarantine"
CORRECTIONS   = "silver_corrections"

print(f"Quarantine in:    {CATALOG}.{QUARANTINE}.qrt_*")
print(f"Manual fixes:     {CATALOG}.{CORRECTIONS}.manual_fixes_*")
print(f"Fix log:          {CATALOG}.{CORRECTIONS}.fix_log")
print(f"Silver target:    {CATALOG}.{SILVER}.dim_*, fact_*")

Quarantine in:    banking.silver_quarantine.qrt_*
Manual fixes:     banking.silver_corrections.manual_fixes_*
Fix log:          banking.silver_corrections.fix_log
Silver target:    banking.silver.dim_*, fact_*


In [0]:
# Helper validate utilizat in toate notebook-urile Silver
def split_valid_invalid(df: DataFrame, validations: list) -> tuple:
    """Aplica validari, returneaza (valid, invalid). Invalid are 'error_reasons' array."""
    error_cols = []
    for rule_name, condition in validations:
        col_name = f"_err_{rule_name}"
        df = df.withColumn(col_name, when(~condition, lit(rule_name)))
        error_cols.append(col_name)
    
    df = df.withColumn(
        "error_reasons",
        array_filter(array(*[col(c) for c in error_cols]), lambda x: x.isNotNull())
    )
    df = df.drop(*error_cols)
    
    df_valid   = df.filter(size(col("error_reasons")) == 0).drop("error_reasons")
    df_invalid = df.filter(size(col("error_reasons")) > 0)
    return df_valid, df_invalid


def add_fix_status_column(qrt_table: str):
    """Asigura ca tabela quarantine are coloana fix_status (default 'pending')."""
    full_name = f"{CATALOG}.{QUARANTINE}.{qrt_table}"
    
    # Verificam daca tabela exista
    if not spark.catalog.tableExists(full_name):
        print(f"  Skip {qrt_table} — tabela nu exista (nimic in quarantine)")
        return
    
    cols = [c.name for c in spark.table(full_name).schema]
    if "fix_status" not in cols:
        spark.sql(f"ALTER TABLE {full_name} ADD COLUMN fix_status STRING")
        spark.sql(f"UPDATE {full_name} SET fix_status = 'pending' WHERE fix_status IS NULL")
        try:
            spark.sql(f"ALTER TABLE {full_name} ALTER COLUMN fix_status SET DEFAULT 'pending'")
        except Exception:
            pass
        print(f"  Adaugat coloana fix_status la {qrt_table}")
    else:
        spark.sql(f"UPDATE {full_name} SET fix_status = 'pending' WHERE fix_status IS NULL")


def append_to_silver(df_fixed: DataFrame, target_table: str, dedup_key: str = None):
    """
    Appendeaza randurile fixate in tabela Silver.
    Daca dedup_key e specificat, eliminam mai intai randurile existente cu acel key
    (pentru a evita duplicate la rerulari).
    """
    target = f"{CATALOG}.{SILVER}.{target_table}"
    
    if dedup_key:
        ids_fixed = [r[0] for r in df_fixed.select(dedup_key).distinct().collect()]
        if ids_fixed:
            delta = DeltaTable.forName(spark, target)
            delta.delete(col(dedup_key).isin(ids_fixed))
    
    # Aliniem schema cu target
    target_cols = spark.table(target).columns
    df_to_write = df_fixed.select(*[
        col(c) if c in df_fixed.columns else lit(None).alias(c)
        for c in target_cols
    ])
    
    df_to_write.write.format("delta").mode("append").saveAsTable(target)


def update_fix_status(qrt_table: str, ids: list, pk_col: str, status: str):
    """Marcheaza randurile din quarantine cu fix_status nou."""
    if not ids:
        return
    delta = DeltaTable.forName(spark, f"{CATALOG}.{QUARANTINE}.{qrt_table}")
    delta.update(
        condition = col(pk_col).isin(ids),
        set = {
            "fix_status": lit(status),
        }
    )

## 2. Inițializare schema corrections + tabele manual_fixes

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS banking.silver_corrections;

In [0]:
# Cream tabelele de manual fixes (daca nu exista)
# Schema generica: pk + column_name + new_value + fixed_by + fixed_at + comment

manual_fixes_schema = """
    pk_value      STRING NOT NULL,
    column_name   STRING NOT NULL,
    new_value     STRING NOT NULL,
    fixed_by      STRING,
    fixed_at      TIMESTAMP,
    comment       STRING
"""

for entity in ["cards", "loans", "transactions", "customers"]:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {CATALOG}.{CORRECTIONS}.manual_fixes_{entity} (
            {manual_fixes_schema}
        ) USING DELTA
    """)

# Tabela de log
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{CORRECTIONS}.fix_log (
        run_id              STRING,
        entity              STRING,
        fix_type            STRING,
        rows_attempted      INT,
        rows_fixed          INT,
        rows_still_invalid  INT,
        run_started_at      TIMESTAMP,
        run_completed_at    TIMESTAMP,
        details_json        STRING
    ) USING DELTA
""")

print("Schema 'silver_corrections' initializata")
print("Tabele create:")
spark.sql(f"SHOW TABLES IN {CATALOG}.{CORRECTIONS}").show(truncate=False)

Schema 'silver_corrections' initializata
Tabele create:
+------------------+-------------------------+-----------+
|database          |tableName                |isTemporary|
+------------------+-------------------------+-----------+
|silver_corrections|fix_log                  |false      |
|silver_corrections|manual_fixes_cards       |false      |
|silver_corrections|manual_fixes_customers   |false      |
|silver_corrections|manual_fixes_loans       |false      |
|silver_corrections|manual_fixes_transactions|false      |
+------------------+-------------------------+-----------+



## 3. Pregătire run

In [0]:
import uuid

RUN_ID = f"fix_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"
RUN_STARTED = datetime.now()

print(f"Run ID: {RUN_ID}")

# Asiguram ca toate qrt_* au coloana fix_status
for qrt in ["qrt_cards", "qrt_loans", "qrt_transactions", "qrt_customers"]:
    try:
        add_fix_status_column(qrt)
    except Exception as e:
        print(f"  WARN {qrt}: {e}")

Run ID: fix_20260502_214019_2f86f603
  Skip qrt_customers — tabela nu exista (nimic in quarantine)


## 4. Funcție de logging

In [0]:
def log_fix_run(entity: str, fix_type: str, attempted: int, fixed: int, still_invalid: int, details: dict = None):
    """Loghează rezultatul unui pas de fix."""
    import json
    
    log_schema = StructType([
        StructField("run_id",             StringType(),    True),
        StructField("entity",              StringType(),    True),
        StructField("fix_type",            StringType(),    True),
        StructField("rows_attempted",      IntegerType(),   True),
        StructField("rows_fixed",          IntegerType(),   True),
        StructField("rows_still_invalid",  IntegerType(),   True),
        StructField("run_started_at",      TimestampType(), True),
        StructField("run_completed_at",    TimestampType(), True),
        StructField("details_json",        StringType(),    True),
    ])
    
    data = [(
        RUN_ID,
        entity,
        fix_type,
        int(attempted),
        int(fixed),
        int(still_invalid),
        RUN_STARTED,
        datetime.now(),
        json.dumps(details or {}),
    )]
    
    df = spark.createDataFrame(data, schema=log_schema)
    df.write.format("delta").mode("append").saveAsTable(
        f"{CATALOG}.{CORRECTIONS}.fix_log"
    )

## 5. Auto-fix + manual override pentru `qrt_cards`

**Auto-fix-uri:**
- `expired_but_active` → `status = 'EXPIRED'`
- `credit_limit_negative` → `credit_limit = ABS(credit_limit)`

In [0]:
print("=" * 60)
print("CARDS — auto-fix + manual override")
print("=" * 60)
 
qrt = spark.table(f"{CATALOG}.{QUARANTINE}.qrt_cards").filter(col("fix_status") == "pending")
n_pending = qrt.count()
print(f"Pending in qrt_cards: {n_pending}")
 
if n_pending > 0:
    today = current_timestamp().cast("date")
    
    # ── AUTO-FIX ──
    df_fixed = (qrt
        # Fix 1: expired_but_active → status = EXPIRED
        .withColumn("status", 
            when(col("expiry_date") < today, lit("EXPIRED"))
            .otherwise(col("status")))
        # Fix 2: credit_limit_negative → ABS
        .withColumn("credit_limit",
            when(col("credit_limit") < 0, spark_abs(col("credit_limit")))
            .otherwise(col("credit_limit")))
    )
    
    # ── MANUAL OVERRIDE ──
    manual = spark.table(f"{CATALOG}.{CORRECTIONS}.manual_fixes_cards")
    n_manual = manual.count()
    
    if n_manual > 0:
        print(f"  Aplicare {n_manual} manual fixes...")
        # Pivotam manual_fixes pe column_name si aplicam pe df_fixed
        # Pentru simplitate, iteram pe fiecare row de manual_fix
        for row in manual.collect():
            pk_value    = row["pk_value"]
            column_name = row["column_name"]
            new_value   = row["new_value"]
            
            if column_name in df_fixed.columns:
                df_fixed = df_fixed.withColumn(
                    column_name,
                    when(col("card_id") == pk_value, lit(new_value))
                    .otherwise(col(column_name))
                )
    
    # ── REVALIDARE ──
    valid_account_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
                         .select("account_id").collect()]
    
    revalidations = [
        ("pk_null",                col("card_id").isNotNull()),
        ("account_id_null",        col("account_id").isNotNull()),
        ("account_id_invalid",     col("account_id").isin(valid_account_ids)),
        ("status_invalid",         col("status").isin(["ACTIVE", "BLOCKED", "EXPIRED", "CANCELLED"])),
        ("expiry_date_null",       col("expiry_date").isNotNull()),
        ("issued_date_null",       col("issued_date").isNotNull()),
        ("expired_but_active",     ~((col("status") == "ACTIVE") & (col("expiry_date") < today))),
        ("credit_limit_negative",  (col("credit_limit").isNull()) | (col("credit_limit") >= 0)),
    ]
    
    df_valid_after_fix, df_still_invalid = split_valid_invalid(df_fixed, revalidations)
    
    n_fixed   = df_valid_after_fix.count()
    n_unfixed = df_still_invalid.count()
    
    print(f"  Auto-fixate + valide: {n_fixed}")
    print(f"  Inca invalide:         {n_unfixed}")
    
    # ── MIGRARE IN SILVER ──
    if n_fixed > 0:
        # Adaugam silver_loaded_at + lineage  
        df_to_silver = df_valid_after_fix.withColumn("silver_loaded_at", current_timestamp())
        append_to_silver(df_to_silver, "dim_cards", dedup_key="card_id")
        
        # Marcam fix_status in quarantine
        ids_fixed = [r[0] for r in df_valid_after_fix.select("card_id").collect()]
        # Determinam care e auto vs manual
        manual_pks = [r[0] for r in manual.select("pk_value").distinct().collect()] if n_manual > 0 else []
        
        ids_auto   = [i for i in ids_fixed if i not in manual_pks]
        ids_manual = [i for i in ids_fixed if i in manual_pks]
        
        update_fix_status("qrt_cards", ids_auto,   "card_id", "auto_fixed")
        update_fix_status("qrt_cards", ids_manual, "card_id", "manual_fixed")
        
        print(f"  Migrate in dim_cards: {n_fixed}")
        print(f"    auto_fixed:   {len(ids_auto)}")
        print(f"    manual_fixed: {len(ids_manual)}")
    
    # ── LOG ──
    log_fix_run(
        entity="cards",
        fix_type="auto+manual",
        attempted=n_pending,
        fixed=n_fixed if n_pending > 0 else 0,
        still_invalid=n_unfixed if n_pending > 0 else 0,
    )
else:
    print("  Nimic de procesat")

CARDS — auto-fix + manual override
Pending in qrt_cards: 0
  Nimic de procesat


## 6. Auto-fix + manual override pentru `qrt_loans`

**Auto-fix-uri:**
- `end_date_before_start` → `end_date = start_date + (term_months * 30 days)`
- `principal_invalid` (negativ) → `principal_amount = ABS(principal_amount)`

In [0]:
print("=" * 60)
print("LOANS — auto-fix + manual override")
print("=" * 60)

qrt = spark.table(f"{CATALOG}.{QUARANTINE}.qrt_loans").filter(col("fix_status") == "pending")
n_pending = qrt.count()
print(f"Pending in qrt_loans: {n_pending}")

if n_pending > 0:
    # ── AUTO-FIX ──
    df_fixed = (qrt
        # Fix 1: end_date < start_date → recalculam din start_date + term_months
        .withColumn("end_date",
            when(col("end_date") < col("start_date"),
                 expr("date_add(start_date, term_months * 30)"))
            .otherwise(col("end_date")))
        # Fix 2: principal negativ → ABS
        .withColumn("principal_amount",
            when(col("principal_amount") < 0, spark_abs(col("principal_amount")))
            .otherwise(col("principal_amount")))
        # Fix 3: outstanding negativ → 0
        .withColumn("outstanding_balance",
            when(col("outstanding_balance") < 0, lit(0.0))
            .otherwise(col("outstanding_balance")))
    )
    
    # ── MANUAL OVERRIDE ──
    manual = spark.table(f"{CATALOG}.{CORRECTIONS}.manual_fixes_loans")
    n_manual = manual.count()
    
    if n_manual > 0:
        print(f"  Aplicare {n_manual} manual fixes...")
        for row in manual.collect():
            pk_value    = row["pk_value"]
            column_name = row["column_name"]
            new_value   = row["new_value"]
            
            if column_name in df_fixed.columns:
                df_fixed = df_fixed.withColumn(
                    column_name,
                    when(col("loan_id") == pk_value, lit(new_value))
                    .otherwise(col(column_name))
                )
    
    # ── REVALIDARE ──
    valid_account_ids = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
                         .select("account_id").collect()]
    
    revalidations = [
        ("pk_null",                  col("loan_id").isNotNull()),
        ("account_id_null",          col("account_id").isNotNull()),
        ("account_id_invalid",       col("account_id").isin(valid_account_ids)),
        ("status_invalid",           col("status").isin(
            ["APPLIED", "APPROVED", "ACTIVE", "OVERDUE", "CLOSED", "REJECTED"])),
        ("principal_invalid",        col("principal_amount") > 0),
        ("end_date_before_start",    col("end_date") >= col("start_date")),
        ("term_invalid",             col("term_months") > 0),
        ("rate_negative",            col("interest_rate_pa") >= 0),
        ("outstanding_invalid",      col("outstanding_balance") >= 0),
    ]
    
    df_valid_after_fix, df_still_invalid = split_valid_invalid(df_fixed, revalidations)
    
    n_fixed   = df_valid_after_fix.count()
    n_unfixed = df_still_invalid.count()
    
    print(f"  Auto-fixate + valide: {n_fixed}")
    print(f"  Inca invalide:         {n_unfixed}")
    
    # ── MIGRARE ──
    if n_fixed > 0:
        df_to_silver = df_valid_after_fix.withColumn("silver_loaded_at", current_timestamp())
        append_to_silver(df_to_silver, "dim_loans", dedup_key="loan_id")
        
        ids_fixed  = [r[0] for r in df_valid_after_fix.select("loan_id").collect()]
        manual_pks = [r[0] for r in manual.select("pk_value").distinct().collect()] if n_manual > 0 else []
        ids_auto   = [i for i in ids_fixed if i not in manual_pks]
        ids_manual = [i for i in ids_fixed if i in manual_pks]
        
        update_fix_status("qrt_loans", ids_auto,   "loan_id", "auto_fixed")
        update_fix_status("qrt_loans", ids_manual, "loan_id", "manual_fixed")
        
        print(f"  Migrate in dim_loans: {n_fixed}")
        print(f"    auto_fixed:   {len(ids_auto)}")
        print(f"    manual_fixed: {len(ids_manual)}")
    
    log_fix_run("loans", "auto+manual", n_pending, n_fixed, n_unfixed)
else:
    print("  Nimic de procesat")

LOANS — auto-fix + manual override
Pending in qrt_loans: 0
  Nimic de procesat


## 7. Auto-fix + manual override pentru `qrt_transactions`

**Auto-fix-uri (cele mai multe — quarantine voluminos):**
- `amount_positive` (negativ) → `amount = ABS(amount)`, `amount_ron = ABS(amount_ron)`
- `completed_before_initiated` → `completed_at = initiated_at + 1 minute`
- `amount_ron_inconsistent` → `amount_ron = amount * exchange_rate`

**Status / channel / type invalid:** se fac doar prin manual override (nu putem ghici).

In [0]:
print("=" * 60)
print("TRANSACTIONS — auto-fix + manual override")
print("=" * 60)

qrt = spark.table(f"{CATALOG}.{QUARANTINE}.qrt_transactions").filter(col("fix_status") == "pending")
n_pending = qrt.count()
print(f"Pending in qrt_transactions: {n_pending}")

if n_pending > 0:
    # ── AUTO-FIX ──
    df_fixed = (qrt
        # Fix 1: amount negativ → ABS
        .withColumn("amount",
            when(col("amount") < 0, spark_abs(col("amount")))
            .otherwise(col("amount")))
        .withColumn("amount_ron",
            when(col("amount_ron") < 0, spark_abs(col("amount_ron")))
            .otherwise(col("amount_ron")))
        # Fix 2: completed_at < initiated_at → initiated_at + 1 min
        .withColumn("completed_at",
            when(
                col("completed_at").isNotNull() & 
                (col("completed_at") < col("initiated_at")),
                expr("initiated_at + INTERVAL 1 MINUTE")
            ).otherwise(col("completed_at")))
        # Fix 3: amount_ron inconsistent → recalc din amount * exchange_rate
        .withColumn("amount_ron",
            when(
                col("amount").isNotNull() & 
                col("exchange_rate").isNotNull() &
                (spark_abs(col("amount_ron") - col("amount") * col("exchange_rate")) > 0.05),
                col("amount") * col("exchange_rate")
            ).otherwise(col("amount_ron")))
    )
    
    # ── MANUAL OVERRIDE ──
    manual = spark.table(f"{CATALOG}.{CORRECTIONS}.manual_fixes_transactions")
    n_manual = manual.count()
    
    if n_manual > 0:
        print(f"  Aplicare {n_manual} manual fixes...")
        for row in manual.collect():
            pk_value    = row["pk_value"]
            column_name = row["column_name"]
            new_value   = row["new_value"]
            
            if column_name in df_fixed.columns:
                df_fixed = df_fixed.withColumn(
                    column_name,
                    when(col("transaction_id") == pk_value, lit(new_value))
                    .otherwise(col(column_name))
                )
    
    # ── REVALIDARE ──
    valid_status_codes  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_transaction_statuses")
                           .select("status_code").collect()]
    valid_channel_codes = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_channels")
                           .select("channel_code").collect()]
    valid_type_codes    = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_transaction_types")
                           .select("type_code").collect()]
    valid_currencies    = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_currencies")
                           .select("currency_code").collect()]
    valid_account_ids   = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_accounts")
                           .select("account_id").collect()]
    valid_card_ids      = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_cards")
                           .select("card_id").collect()]
    valid_employee_ids  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.dim_employees")
                           .select("employee_id").collect()]
    
    TOL = 0.05
    revalidations = [
        ("pk_null",              col("transaction_id").isNotNull()),
        ("account_id_null",      col("account_id").isNotNull()),
        ("amount_null",          col("amount").isNotNull()),
        ("amount_positive",      (col("amount").isNull()) | (col("amount") > 0)),
        ("account_id_invalid",   col("account_id").isin(valid_account_ids)),
        ("status_invalid",       col("status_code").isin(valid_status_codes)),
        ("channel_invalid",      col("channel_code").isin(valid_channel_codes)),
        ("type_invalid",         col("type_code").isin(valid_type_codes)),
        ("currency_invalid",     col("currency_code").isin(valid_currencies)),
        ("card_id_invalid", 
            (col("card_id").isNull()) | (col("card_id").isin(valid_card_ids))),
        ("employee_id_invalid", 
            (col("employee_id").isNull()) | (col("employee_id").isin(valid_employee_ids))),
        ("completed_before_initiated", 
            (col("completed_at").isNull()) | (col("completed_at") >= col("initiated_at"))),
        ("amount_ron_inconsistent",
            (col("amount").isNull()) | (col("amount_ron").isNull()) | (col("exchange_rate").isNull()) |
            (spark_abs(col("amount_ron") - col("amount") * col("exchange_rate")) <= lit(TOL))),
    ]
    
    df_valid_after_fix, df_still_invalid = split_valid_invalid(df_fixed, revalidations)
    
    n_fixed   = df_valid_after_fix.count()
    n_unfixed = df_still_invalid.count()
    
    print(f"  Auto-fixate + valide: {n_fixed}")
    print(f"  Inca invalide:         {n_unfixed}")
    
    # ── MIGRARE IN fact_transactions ──
    # Atentie: fact_transactions are coloane suplimentare (customer_sk, customer_id, date_id)
    # care nu exista in quarantine. Trebuie sa le calculam.
    if n_fixed > 0:
        # JOIN cu dim_accounts pentru customer_id
        df_with_cust = df_valid_after_fix.alias("tx").join(
            spark.table(f"{CATALOG}.{SILVER}.dim_accounts").select("account_id", "customer_id").alias("acc"),
            col("tx.account_id") == col("acc.account_id"),
            "left"
        ).select("tx.*", col("acc.customer_id").alias("customer_id"))
        
        # JOIN SCD2 cu dim_customers pentru customer_sk
        dim_customers = spark.table(f"{CATALOG}.{SILVER}.dim_customers").select(
            col("customer_id").alias("dim_customer_id"),
            col("customer_sk"),
            col("valid_from"),
            col("valid_to"),
        )
        
        df_with_sk = df_with_cust.alias("tx").join(
            dim_customers.alias("dim"),
            (col("tx.customer_id") == col("dim.dim_customer_id")) &
            (col("dim.valid_from") <= col("tx.initiated_at")) &
            (col("dim.valid_to") > col("tx.initiated_at")),
            "left"
        ).select("tx.*", col("dim.customer_sk").alias("customer_sk"))
        
        # date_id
        from pyspark.sql.functions import year as f_year, month as f_month, dayofmonth as f_day
        df_with_sk = df_with_sk.withColumn(
            "date_id",
            (f_year("initiated_at") * 10000 + f_month("initiated_at") * 100 + f_day("initiated_at")).cast(IntegerType())
        )
        
        df_to_silver = df_with_sk.withColumn("silver_loaded_at", current_timestamp())
        append_to_silver(df_to_silver, "fact_transactions", dedup_key="transaction_id")
        
        # Marcam fix_status
        ids_fixed  = [r[0] for r in df_valid_after_fix.select("transaction_id").collect()]
        manual_pks = [r[0] for r in manual.select("pk_value").distinct().collect()] if n_manual > 0 else []
        ids_auto   = [i for i in ids_fixed if i not in manual_pks]
        ids_manual = [i for i in ids_fixed if i in manual_pks]
        
        update_fix_status("qrt_transactions", ids_auto,   "transaction_id", "auto_fixed")
        update_fix_status("qrt_transactions", ids_manual, "transaction_id", "manual_fixed")
        
        print(f"  Migrate in fact_transactions: {n_fixed}")
        print(f"    auto_fixed:   {len(ids_auto)}")
        print(f"    manual_fixed: {len(ids_manual)}")
    
    log_fix_run("transactions", "auto+manual", n_pending, n_fixed, n_unfixed)
else:
    print("  Nimic de procesat")

TRANSACTIONS — auto-fix + manual override
Pending in qrt_transactions: 3610
  Auto-fixate + valide: 2847
  Inca invalide:         763
  Migrate in fact_transactions: 2847
    auto_fixed:   2847
    manual_fixed: 0


## 8. Auto-fix + manual override pentru `qrt_customers`

**Auto-fix-uri (limitate — majoritatea erorilor pe customers necesită override manual):**
- `cnp_length` (CNP cu 12 caractere) — neauto-fixabil (nu putem ghici cifra lipsă)
- `email_format` — auto-fix limitat (adăugare @domeniu.placeholder)

Pentru customers, majoritatea cazurilor merg pe **manual override** (analist KYC verifică și introduce valorile corecte).

In [0]:
print("=" * 60)
print("CUSTOMERS — auto-fix + manual override")
print("=" * 60)

qrt = spark.table(f"{CATALOG}.{QUARANTINE}.qrt_customers").filter(col("fix_status") == "pending")
n_pending = qrt.count()
print(f"Pending in qrt_customers: {n_pending}")

if n_pending > 0:
    df_fixed = qrt
    
    # ── MANUAL OVERRIDE (principala metoda pentru customers) ──
    manual = spark.table(f"{CATALOG}.{CORRECTIONS}.manual_fixes_customers")
    n_manual = manual.count()
    
    if n_manual > 0:
        print(f"  Aplicare {n_manual} manual fixes...")
        for row in manual.collect():
            pk_value    = row["pk_value"]
            column_name = row["column_name"]
            new_value   = row["new_value"]
            
            if column_name in df_fixed.columns:
                df_fixed = df_fixed.withColumn(
                    column_name,
                    when(col("customer_id") == pk_value, lit(new_value))
                    .otherwise(col(column_name))
                )
    
    # ── REVALIDARE ──
    valid_counties  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_counties")
                       .select("county_code").collect()]
    valid_segments  = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_customer_segments")
                       .select("segment_code").collect()]
    valid_kyc       = [r[0] for r in spark.table(f"{CATALOG}.{SILVER}.ref_kyc_statuses")
                       .select("status_code").collect()]
    
    revalidations = [
        ("pk_null",            col("customer_id").isNotNull()),
        ("cnp_null",           col("cnp").isNotNull()),
        ("cnp_length",         (col("cnp").isNull()) | (col("cnp").rlike("^[0-9]{13}$"))),
        ("email_null",         col("email").isNotNull()),
        ("email_format",       (col("email").isNull()) | (col("email").rlike(".+@.+\\..+"))),
        ("dob_null",           col("date_of_birth").isNotNull()),
        ("gender_invalid",     col("gender").isin(["M", "F"])),
        ("county_invalid",     col("county_code").isin(valid_counties)),
        ("segment_invalid",    col("segment_code").isin(valid_segments)),
        ("kyc_invalid",        col("kyc_status").isin(valid_kyc)),
    ]
    
    df_valid_after_fix, df_still_invalid = split_valid_invalid(df_fixed, revalidations)
    
    n_fixed   = df_valid_after_fix.count()
    n_unfixed = df_still_invalid.count()
    
    print(f"  Manual fixate + valide: {n_fixed}")
    print(f"  Inca invalide:           {n_unfixed}")
    
    # ── MIGRARE IN dim_customers (cu SCD2 — INSERT versiune noua daca client nou) ──
    if n_fixed > 0:
        # Verificam care sunt clienti complet noi vs care exista deja
        existing_customer_ids = [r[0] for r in 
            spark.table(f"{CATALOG}.{SILVER}.dim_customers").select("customer_id").distinct().collect()]
        
        df_new = df_valid_after_fix.filter(~col("customer_id").isin(existing_customer_ids))
        n_new = df_new.count()
        
        if n_new > 0:
            from pyspark.sql.functions import sha2, coalesce
            from pyspark.sql.window import Window
            from pyspark.sql.functions import row_number, max as spark_max
            
            SCD_TRACKED = ["segment_code", "kyc_status", "county_code", "is_active"]
            
            df_new = df_new.withColumn(
                "scd_hash",
                sha2(concat_ws("|", *[coalesce(col(c).cast("string"), lit("__NULL__")) for c in SCD_TRACKED]), 256)
            )
            
            now = current_timestamp()
            df_new = (df_new
                .withColumn("valid_from",       to_timestamp(lit("1900-01-01 00:00:00")))
                .withColumn("valid_to",         to_timestamp(lit("3000-01-01 00:00:00")))
                .withColumn("is_current",       lit(True))
                .withColumn("silver_loaded_at", now)
            )
            
            # Generam customer_sk continuand de la max existent
            silver_dim = spark.table(f"{CATALOG}.{SILVER}.dim_customers")
            max_sk = silver_dim.agg(spark_max("customer_sk")).collect()[0][0] or 0
            window = Window.orderBy("customer_id")
            df_new = df_new.withColumn("customer_sk", row_number().over(window) + lit(max_sk))
            
            # Aliniem schema cu target
            target_cols = silver_dim.columns
            df_new = df_new.select(*[
                col(c) if c in df_new.columns else lit(None).alias(c)
                for c in target_cols
            ])
            
            df_new.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SILVER}.dim_customers")
            print(f"  Inserat in dim_customers: {n_new} clienti noi (SCD2 v1)")
        
        # Marcam fix_status
        ids_fixed  = [r[0] for r in df_valid_after_fix.select("customer_id").collect()]
        manual_pks = [r[0] for r in manual.select("pk_value").distinct().collect()] if n_manual > 0 else []
        ids_auto   = [i for i in ids_fixed if i not in manual_pks]
        ids_manual = [i for i in ids_fixed if i in manual_pks]
        
        update_fix_status("qrt_customers", ids_auto,   "customer_id", "auto_fixed")
        update_fix_status("qrt_customers", ids_manual, "customer_id", "manual_fixed")
        
        print(f"  Total marcate in quarantine: {n_fixed}")
    
    log_fix_run("customers", "auto+manual", n_pending, n_fixed, n_unfixed)
else:
    print("  Nimic de procesat")

CUSTOMERS — auto-fix + manual override


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5078549288788963>, line 6
      3 print("=" * 60)
      5 qrt = spark.table(f"{CATALOG}.{QUARANTINE}.qrt_customers").filter(col("fix_status") == "pending")
----> 6 n_pending = qrt.count()
      7 print(f"Pending in qrt_customers: {n_pending}")
      9 if n_pending > 0:

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/dataframe.py:300, in DataFrame.count(self)
    299 def count(self) -> int:
--> 300     table, _ = self.agg(F._invoke_function("count", F.lit(1)))._to_table()
    301     return table[0][0].as_py()

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/dataframe.py:1971, in DataFrame._to_table(self)
   1969 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1970     query = self._plan.to_proto(self._session.client)
-> 1971     table, schema, self._exec

## 9. Raport global — distribuția fix_status în quarantine

In [0]:
%sql
SELECT 
    'cards' AS entity,
    fix_status,
    COUNT(*) AS n_rows
FROM banking.silver_quarantine.qrt_cards
GROUP BY fix_status
UNION ALL
SELECT 'loans', fix_status, COUNT(*)
FROM banking.silver_quarantine.qrt_loans
GROUP BY fix_status
UNION ALL
SELECT 'transactions', fix_status, COUNT(*)
FROM banking.silver_quarantine.qrt_transactions
GROUP BY fix_status
UNION ALL
SELECT 'customers', fix_status, COUNT(*)
FROM banking.silver_quarantine.qrt_customers
GROUP BY fix_status
ORDER BY entity, fix_status;

entity,fix_status,n_rows
cards,auto_fixed,14
loans,auto_fixed,5
transactions,pending,763
transactions,auto_fixed,2847


## 10. Fix log — istoricul rulărilor

In [0]:
%sql
SELECT 
    run_id,
    entity,
    rows_attempted,
    rows_fixed,
    rows_still_invalid,
    ROUND(rows_fixed * 100.0 / NULLIF(rows_attempted, 0), 1) AS fix_rate_pct,
    run_completed_at
FROM banking.silver_corrections.fix_log
ORDER BY run_completed_at DESC
LIMIT 20;

run_id,entity,rows_attempted,rows_fixed,rows_still_invalid,fix_rate_pct,run_completed_at
fix_20260502_214019_2f86f603,transactions,3610,2847,763,78.9,2026-05-02T21:44:18.899Z


## 11. Demonstrare — exemplu de manual override

Dacă vrei să adaugi un manual fix, folosești INSERT direct în tabela `manual_fixes_*`.
La următoarea rulare a notebook-ului, fix-ul va fi aplicat automat.

In [0]:
%sql
-- Exemplu: vezi ce tranzactii au ramas in quarantine cu status invalid
SELECT 
    transaction_id, 
    status_code, 
    channel_code,
    error_reasons_str,
    fix_status
FROM banking.silver_quarantine.qrt_transactions
WHERE fix_status = 'pending'
  AND array_contains(error_reasons, 'status_invalid')
LIMIT 5;

transaction_id,status_code,channel_code,error_reasons_str,fix_status
TXN-00000199,INVALID_STATUS,MOBILE,status_invalid,pending
TXN-00000269,INVALID_STATUS,ATM,status_invalid,pending
TXN-00000710,INVALID_STATUS,ATM,status_invalid,pending
TXN-00000816,INVALID_STATUS,ONLINE,status_invalid,pending
TXN-00001223,INVALID_STATUS,POS,status_invalid,pending


In [0]:
%sql
--Exemplu de inserare manuala (descomenteaza si modifica pk_value cu un id real)
INSERT INTO banking.silver_corrections.manual_fixes_transactions VALUES
    ('TXN-00000816', 'status_code', 'COMPLETED', 'analyst.test', current_timestamp(), 'Validat in sistemul OLTP, status corect e COMPLETED');

num_affected_rows,num_inserted_rows
1,1


## Sumar 02d — Silver Corrections

Realizat:
- Schema `silver_corrections` cu 4 tabele `manual_fixes_*` + `fix_log`
- Coloana `fix_status` adăugată în toate `qrt_*` (default `pending`)
- **Auto-fix-uri** pentru cele 4 entități:
  - cards: expired+active, credit_limit negativ
  - loans: end_date < start_date, sume negative
  - transactions: amount negativ, completed_at inversat, amount_ron inconsistent
  - customers: doar manual override (cele mai sensibile)
- **Manual override** prin INSERT în `manual_fixes_*`
- **Migrare în Silver** cu dedup și completarea coloanelor lipsă (customer_sk, date_id, etc.)
- **Audit trail complet** prin `fix_log` — fiecare rulare e logată

**Workflow recomandat:**
1. Rulezi `02d` — auto-fix se aplică, marcează tot ce poate
2. Analistul examinează `qrt_*` cu `fix_status = 'pending'`
3. Analistul scrie `INSERT INTO manual_fixes_*` cu corecțiile
4. Rerulezi `02d` — manual fixes se aplică
5. Repeti până când quarantine este aproape gol

**Următorul pas — Gold!**